<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day22_practice3_agent_service_%EC%99%84%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# agent_service 완성 - 판단하는 Agent
# 실습 1, 2 도구 루프는 Agent의 일반형(챗봇 확장 방향)이고
# 서버의 신고 파이프라인 Agent는 한 번의 종합 판단에 집중
# YOLO 감지 + 멀티 모달 해석 + RAG 규정을 종합해서
# 신고해야 하는가(should_report), 안내문(guidance), 필요하면 신고서 초안(report_draft 6필드) 자동 작성)

In [2]:
import os, requests, json, re
import torch
import torch.nn.functional as F

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()
# API 호출 - openAI 호환 api 형식으로 코드 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환
MODEL = "gemini-3.5-flash-lite"

In [3]:
USE_REAL_AI = bool(API_KEY)
FALLBACK_ON_ERROR = True # real이 실패해도 mock으로 대체할지
REPORT_THRESHOLD = 3 # 심각도 3 이상이면 '신고 권장'으로 판단

def _as_int(v, default = 3, lo = 1, hi = 5): # 1~5 정수로
  try:
    n = int(float(v))
  except (TypeError, ValueError):
    return default
  return max(lo, min(hi, n))

```
{
  "should_report": true/false,
  "guidance": "사용자 안내문 (2~3문장)",
  "report_draft": {{"hazard_type": "...", "location": "...",
    "severity": 1~5 사이 정수 (반드시 숫자로), "description": "...",
    "regulation": "근거 규정", "recommended_action": "권고 조치"}}
  }
신고가 불필요하면 report_draft는 null
```

In [4]:
# 셀 1. _normalize - '신고서 6필드'를 항상 채운다(서버와 동일)
def _normalize(result, detections, analysis, regulations):
  if not isinstance(result, dict):
    result = {}
  analysis = analysis or {} # None 방어
  regulations = regulations or []
  should_report = bool(result.get("should_report", False))
  guidance = str(result.get("guidance") or "안전 운전하세요.")
  draft = result.get("report_draft")
  if should_report and isinstance(draft, dict): # '신고'이고 초안이 dict일 때만 신고서 6필드 채움

    top_type = ""
    if detections: # max(리스트, key=함수) 리스트 각 원소에 함수를 적용한 값이 가장 큰 원소를 반환
      top = max(detections, key = lambda d: _as_int(d.get("severity"), 0, lo=0)) # 심각도가 가장 높은 감지 항목 한 개 {"type": "포트홀", "severity":4}
      top_type = str(top.get("type", "")) # "포트홀"

    reg_title = regulations[0].get("title", "관련 규정") if regulations else "관련 규정" # 규정 제목, "도로법 시행령 제 55조"

    report_draft = { # 앱 UI가 6개 필드를 모두 요구
        "hazard_type": str(draft.get("hazard_type") or top_type or "도로 위험"),
        "location": str(draft.get("location") or analysis.get("location_hint") or "위치 확인 중"),
        "severity": _as_int(draft.get("severity"), default=_as_int(analysis.get("ai_severity"), 3)),
        "description": str(draft.get("description") or analysis.get("description") or ""),
        "regulation": str(draft.get("regulation") or reg_title),
        "recommended_action": str(draft.get("recommended_action") or "관할 기관 확인 후 조치 권장"),
    }
  else:
    report_draft = None # 신고 불필요, 초안 없음
  return {"guidance": guidance, "report_draft": report_draft, "should_report": should_report}

In [5]:
# 셀 2. mock - 규칙 기반 판단
def _mock_generate(detections, analysis, regulations):
  pass

In [6]:
# 셀 3. _real_generate - LLM이 '판단'한다
def _real_generate(detections, analysis, regulations):
  from openai import OpenAI # 제미나이도 OpenAI SDK로 호출(호환성)
  client = OpenAI(api_key = API_KEY, base_url=GEMINI_BASE_URL)

  analysis = analysis or {} # AttributeError 방지
  regulations = regulations or [] # TypeError 방지
  # 프롬프트에는 한 줄 문장으로 넣어야 LLM이 잘 이해하고 토큰도 아낀다
  detected = ", ".join(f"{d.get('type', '위험')}(심각도{d.get('severity', '?')})" for d in detections or []) # "포트홀(심각도 4), 맨홀(심각도 5)"
  reg_text = "\n".join(f"- {r.get('title', '')}: {r.get('content', '')}" for r in regulations) # "- 도로법 제52조: 도로관리청은...\n..."

  prompt = f"""당신은 도로 위험 대응 AI Agent입니다.
  아래 정보를 종합해 판단하고 대응하세요.
  [감지된 위험] {detected}
  [상황 해석] {analysis.get('description', '')}
  [관련 규정]
  {reg_text}

  다음을 판단해 JSON으로만 답하세요:
  1. 이 위험을 신고해야 하는가? (심각도·위험성 고려) "severity"가 2이하이면 신고하니 않는다.
  2. 사용자에게 줄 안내문 (행동 요령)
  3. 신고가 필요하면 신고서 초안

  {{
  "should_report": true/false,
  "guidance": "사용자 안내문 (2~3문장)",
  "report_draft": {{"hazard_type": "...", "location": "...",
    "severity": 1~5 사이 정수 (반드시 숫자로), "description": "...",
    "regulation": "근거 규정", "recommended_action": "권고 조치"}}
  }}
  신고가 불필요하면 report_draft는 null. """
  # {변수}는 값으로 치환되고, {{}}는 중괄호 글자 하나로 출력된다.

  response = client.chat.completions.create(model=MODEL, # LLM 호출(OpenAI 호환성)
                                            messages=[{"role":"user", "content":prompt}], response_format={"type":"json_object"}, max_tokens=2048)
  try:
    return json.loads(response.choices[0].message.content)
  except (json.JSONDecodeError, TypeError):
    return {}

In [7]:
# 셀 4. 공개 함수
def generate_response(detections, analysis, regulations):
  if not USE_REAL_AI:
    return _normalize(_mock_generate(detections, analysis, regulations), detections, analysis, regulations)
  try:
    return _normalize(_real_generate(detections, analysis, regulations), detections, analysis, regulations)
  except Exception as e:
    print(f"[Agent] 생성 실패: {type(e).__name__}: {e}")
    if not FALLBACK_ON_ERROR:
      raise # 폭백 끄면 예외 그대로
    print("[Agent] 가짜 생성으로 대체합니다")
    return _normalize(_mock_generate(detections, analysis, regulations), detections, analysis, regulations)

print(f"모드: {"REAL(Gemini)" if USE_REAL_AI else "MOCK"}")
# 파이프라인 입력
감지 = [{"type": "포트홀", "confidence": 0.92, "severity": 4, # YOLO 모델 결과
       "box": {"x": 0.15, "y": 0.55, "w":0.25, "h": 0.20}}]
해석 = {"description": "도로 중앙에 깊은 포트홀이 있어 차량 손상 위험이 큽니다.", # 멀티모달 결과
      "ai_severity": 4, "location_hint": "도심 2차선", "risk_factors": ["차량 손상"]}
규정 = [{"title": "도로법 시행령 제 55조", "source": "도로법", # RAG 결과
       "content": "포트홀 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다"}]
r = generate_response(감지, 해석, 규정)

모드: REAL(Gemini)


In [8]:
# 셀 5. 시나리오 검증
# 시나리오1 : 심각도 4 → 신고 + 신고서 초안 6필드
r = generate_response(감지, 해석, 규정)
assert set(r.keys()) == {"guidance", "report_draft", "should_report"}
assert r["should_report"] is True and isinstance(r["report_draft"], dict)
assert set(r["report_draft"].keys()) == {"hazard_type", "location", "severity", "description", "regulation", "recommended_action"}
print(r)

{'guidance': '도로 중앙에 심각한 포트홀이 감지되었습니다. 차량 파손 및 사고 위험이 높으니 해당 차로를 피하여 서행하시기 바랍니다.', 'report_draft': {'hazard_type': '포트홀', 'location': '도로 중앙', 'severity': 4, 'description': '도로 중앙에 깊은 포트홀이 발생하여 통행 차량의 타이어 및 서스펜션 손상 위험이 큼.', 'regulation': '도로법 시행령 제 55조', 'recommended_action': '24시간 이내 응급 복구 조치 시행'}, 'should_report': True}


In [9]:
# 시나리오2 : 심각도 2 → 안내만, 신고서 초안 None
r2 = generate_response([{"type":"균열", "severity":2, "confidence":0.7,
                         "box": {"x":0.1, "y":0.1,"w":0.1,"h":0.1}}],
                       {"description":"미세 균열", "ai_severity":2,
                        "location_hint":"이면도로", "risk_factors":[]}, 규정)
assert r2["should_report"] is False and r2["report_draft"] is None
print(r2)

{'guidance': '현재 감지된 위험은 미세 균열로 심각도가 낮아 즉각적인 신고 대상은 아닙니다. 다만, 주행 시 주의하시고 균열이 확대될 경우 별도로 신고해 주시기 바랍니다.', 'report_draft': None, 'should_report': False}


In [10]:
# 시나리오3 : 오염 방어 → LLM이 severity를 문자열로 주고 필드를 빠뜨려도
오염 = {"should_report": True, "guidance": "신고하세요", "report_draft": {"severity":"5"}} # LLM응답
r3 = _normalize(오염, 감지, 해석, 규정)
assert r3["report_draft"]["severity"] == 5
assert r3["report_draft"]["regulation"] == "도로법 시행령 제 55조"
assert r3["report_draft"]["location"] == "도심 2차선"
print(r3)

{'guidance': '신고하세요', 'report_draft': {'hazard_type': '포트홀', 'location': '도심 2차선', 'severity': 5, 'description': '도로 중앙에 깊은 포트홀이 있어 차량 손상 위험이 큽니다.', 'regulation': '도로법 시행령 제 55조', 'recommended_action': '관할 기관 확인 후 조치 권장'}, 'should_report': True}
